# FINA 4075/5075 — Project 2 · The Strategy Lab — checkpoint-2 rerun (data through October 2026)

**Repo:** FINTECH-P2-Example · **Track 1:** intermediate momentum, months t−12 … t−7 (top 20 of 200, equal weights, 10 bp per side) · **Track 2:** frozen prompt on Claude Fable 5.1 Max · **Pre-registration commit:** 293cd99 (9/15/2026; last pre-deadline commit 0e9a61c) · **This file:** the 9/20 notebook rerun for checkpoint 2 (Sun 11/8/2026), saved under a dated name so the grader can diff it against the committed original and the checkpoint-1 copy (`FINA4075_P2_StrategyLab_2026-09-30.ipynb`).

**What changed, and what did not.** Cell 2 points at the dated extension file `panel_returns_2026-10-31.csv` (SHA-256 from the Data Dictionary; its new row is dated 2026-10-30, the last trading day of October); the in-sample window in Cell 8 stays pinned to January 2020 – August 2026 so no live month enters the backtest; Cells 12–16 are the checkpoint routine (Step 10): rerun check against checkpoint 1's Track 1 list, October scored with the live record so far, November holdings, placebo after two months, Track 2 paste. QRVO (Qorvo) was acquired on 10/5/2026: the extension file carries its final partial-month return for October, and it leaves the universe from November (Data Dictionary, Section 4). **`strategy.py` is byte-identical to the 9/20 commit** (Cell 6 rewrites the same text; the Code Rule forbids any edit).

**Before running anything:** File → **Save a copy in Drive**, then Runtime → Change runtime type → **Runtime version 2026.07** (Rule 0). Run top to bottom; finish with **Runtime → Restart session and run all** (Rule 6) before you save the notebook to GitHub.

*Executed copy, 11/2/2026:* every cell run top to bottom with the course's pinned package versions (pandas 2.2.2, numpy 2.0.2, matplotlib 3.10.0, yfinance 0.2.66) on Python 3.12; the build string printed by Cell 4 is the executing machine's, not Colab's. Opening this file in Colab and choosing Runtime → Restart session and run all reproduces every number. The one inline figure preview (the growth-of-$10,000 chart) is omitted from this committed copy to keep the file small; the chart is saved to outputs/ by Cell 9 and regenerates on Run all.


In [1]:
# Cell 1 — mount Drive (Rule 1)
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
# Cell 2 — course folder and the packet files (Rule 1 / Rule 4: analysis reads the dated snapshot)
import os
P2 = "/content/drive/MyDrive/FINA4075/P2"
OUT = P2 + "/outputs"
os.makedirs(OUT, exist_ok=True)

PANEL_FILE = P2 + "/panel_returns_2026-10-31.csv"      # checkpoint 2: the dated extension file (the 93 rows of the 9/30 file + October 2026)
TICKER_FILE = P2 + "/P2_ticker_list_2026-08-31.csv"
PANEL_SHA256 = "f4732d3f86648720cb7de55cb7139ab788da8cd7872da9cad11420a5eac5da27"   # from the Data Dictionary (extension row added 11/2/2026)
print("Folder ready:", P2)


Folder ready: /content/drive/MyDrive/FINA4075/P2


In [3]:
# Cell 3 — pinned setup (Rule 2). Official pinned line from D2L; the P2 analysis itself needs only pandas, numpy and
# matplotlib, which the pinned runtime already provides — never reinstall those.
%pip install -q yfinance==0.2.66

Note: you may need to restart the kernel to use updated packages.


In [4]:
# Cell 4 — environment stamp (Rule 0)
import sys, pandas as pd, numpy as np, matplotlib, yfinance as yf
print(sys.version)
print("pandas", pd.__version__, "| numpy", np.__version__,
      "| matplotlib", matplotlib.__version__, "| yfinance", yf.__version__)

3.12.3 (main, Aug 31 2026, 10:18:26) [GCC 13.3.0]
pandas 2.2.2 | numpy 2.0.2 | matplotlib 3.10.0 | yfinance 0.2.66


## Step 1 — load the packet and verify the hash
The SHA-256 of the panel file must equal the value published in the Data Dictionary. If it does not, the file was edited or re-saved: download it again from D2L.

In [5]:
# Cell 5 — hash check and load
import hashlib
sha = hashlib.sha256(open(PANEL_FILE, "rb").read()).hexdigest()
print("SHA-256:", sha)
assert sha == PANEL_SHA256, "panel file does not match the Data Dictionary hash"

panel = pd.read_csv(PANEL_FILE, index_col=0, parse_dates=True)
info = pd.read_csv(TICKER_FILE).set_index("Ticker")
print("panel shape:", panel.shape, "| first month:", panel.index[0].date(), "| last month:", panel.index[-1].date())
print("missing values:", int(panel.isna().sum().sum()), "| tickers match the company list:", list(panel.columns) == list(info.index))

SHA-256: f4732d3f86648720cb7de55cb7139ab788da8cd7872da9cad11420a5eac5da27
panel shape: (94, 200) | first month: 2019-01-31 | last month: 2026-10-30
missing values: 0 | tickers match the company list: True


## Steps 4 and 7 — the locked rule
The rule lives in one plain-text file, `strategy.py`, so the grader can diff it against the 9/20 commit. Cell 6 writes the file into the P2 folder; Cell 7 imports it. `strategy_rules.md` governs if the two ever disagree.

In [6]:
%%writefile /content/drive/MyDrive/FINA4075/P2/strategy.py
"""
strategy.py — LOCKED STRATEGY CODE (Track 1)   ·   FINTECH-P2-Example   ·   FINA 4075/5075, Fall 2026

Rule (see strategy_rules.md, which governs if the two ever disagree):
  Intermediate momentum, months t-12 through t-7.  For the portfolio held in month t, rank the 200
  stocks by their compounded total return over the SIX monthly returns t-12, t-11, ..., t-7
  (the most recent six months, t-6 .. t-1, are skipped entirely).  Hold the top 20, equal weights,
  rebalanced at every month-end from data through the prior month-end only.  10 bp per side.

CODE RULE: this file is read-only after the 9/20/2026 pre-registration commit.  At each checkpoint
the panel is extended by one month and this code is rerun unchanged.
"""
import numpy as np
import pandas as pd

PER_SIDE = 0.0010        # transaction cost, 10 basis points per side
N_HOLD = 20              # holding count (the top decile of the 200-stock panel)
WINDOW_START = 12        # window starts WINDOW_START months before month t  (t-12)
WINDOW_END = 7           # window ends   WINDOW_END   months before month t  (t-7), inclusive
FIRST_TRADEABLE = WINDOW_START   # the first month t whose window exists is row 12 (January 2020)


def load_panel(panel_path, ticker_path):
    """Read the course packet.  Returns (rets, dates, tickers, info).

    rets    : numpy array, T x N, simple monthly total returns (row 0 = January 2019)
    dates   : the month-end dates of the rows
    tickers : list of the N ticker symbols, in the column order of the panel
    info    : the company list (ticker, company, sector, ...) indexed by ticker
    """
    panel = pd.read_csv(panel_path, index_col=0, parse_dates=True)
    info = pd.read_csv(ticker_path).set_index("Ticker")
    tickers = list(panel.columns)
    assert list(info.index) == tickers, "ticker list and panel columns differ"
    assert not panel.isna().any().any(), "panel has missing values"
    return panel.to_numpy(), panel.index, tickers, info


def signal(t, rets):
    """Compounded return over rows t-12 .. t-7 inclusive (six monthly returns).

    Python slicing excludes the stop index, so rows t-12 .. t-7 are rets[t-12 : t-6].
    """
    window = rets[t - WINDOW_START : t - WINDOW_END + 1]
    assert window.shape[0] == WINDOW_START - WINDOW_END + 1 == 6
    return np.prod(1 + window, axis=0) - 1


def select_holdings(t, rets, tickers):
    """The 20 stocks held during month t: highest signal first; ties broken by alphabetical ticker.

    A stock with any missing return in its t-12..t-7 window is ineligible for month t (rules: MISSING
    DATA). The released panel has no gaps, so every stock is eligible in-sample; this guard governs
    the live window, where an extension file could carry a delisted name's partial history.
    """
    window = rets[t - WINDOW_START : t - WINDOW_END + 1]
    eligible = ~np.isnan(window).any(axis=0)
    sig = signal(t, rets)
    cand = [i for i in range(len(tickers)) if eligible[i]]
    order = sorted(cand, key=lambda i: (-sig[i], tickers[i]))
    return set(order[:N_HOLD])


def run_track(select_fn, rets, t0, t1, per_side=PER_SIDE):
    """Paper-trade a selection rule from month t0 to month t1 (row indices, inclusive).

    Course conventions: equal weights; one-way turnover = names replaced / 20 (the first month
    counts as a full purchase); monthly cost = 2 x per_side x turnover; growth path starts at 1.0.
    Returns (values, turnovers, holdings) with len(values) == months + 1.
    """
    prev, vals, turns, holdings = set(), [1.0], [], []
    for t in range(t0, t1 + 1):
        hold = select_fn(t, rets)
        turn = 1.0 if not prev else len(hold - prev) / N_HOLD
        gross = rets[t, sorted(hold)].mean()
        vals.append(vals[-1] * (1 + gross - 2 * per_side * turn))
        prev = hold
        turns.append(turn)
        holdings.append(hold)
    return np.array(vals), np.array(turns), holdings


def bench(rets, t0, t1):
    """Equal-weighted average of all 200 stocks, rebalanced monthly, no costs."""
    vals = [1.0]
    for t in range(t0, t1 + 1):
        vals.append(vals[-1] * (1 + rets[t].mean()))
    return np.array(vals)


def stats(vals, turns=None):
    """CAGR, annualized volatility, Sharpe (rf = 0), maximum drawdown, annual one-way turnover."""
    months = len(vals) - 1
    monthly = vals[1:] / vals[:-1] - 1
    cagr = vals[-1] ** (12 / months) - 1
    vol = monthly.std(ddof=1) * np.sqrt(12)
    maxdd = (vals / np.maximum.accumulate(vals) - 1).min()
    out = {"CAGR": cagr, "Annualized volatility": vol, "Sharpe (rf = 0)": cagr / vol,
           "Maximum drawdown": maxdd, "Growth of $10,000": 10000 * vals[-1]}
    if turns is not None:
        out["Annual one-way turnover"] = turns.mean() * 12
        out["Cost drag per year"] = 2 * PER_SIDE * turns.mean() * 12
    return out


Writing /content/drive/MyDrive/FINA4075/P2/strategy.py


In [7]:
# Cell 7 — import the locked code from Drive and bind the rule to this panel
import importlib, sys
if P2 not in sys.path:
    sys.path.insert(0, P2)
import strategy as S
importlib.reload(S)

rets, dates, tickers, info = S.load_panel(PANEL_FILE, TICKER_FILE)
def rule(t, rets):
    return S.select_holdings(t, rets, tickers)
print("rows (months):", len(dates), "| first tradeable row:", S.FIRST_TRADEABLE, "=", dates[S.FIRST_TRADEABLE].date())

rows (months): 94 | first tradeable row: 12 = 2020-01-31


## Step 7 — in-sample backtest, January 2020 – August 2026 (80 months)
Pre-live sample only. Net of 10 bp per side; the benchmark is the equal-weighted panel with no costs; rf = 0 for the Sharpe ratio.

In [8]:
# Cell 8 — run both tracks' in-sample paths and the stats table
t0, t1 = S.FIRST_TRADEABLE, 91                           # row 12 (Jan 2020) .. row 91 (Aug 2026): the pre-registered in-sample window; live months (row 92 on) are scored at checkpoints, never backtested
assert dates[t1].strftime('%Y-%m') == '2026-08', 'in-sample window must end in August 2026'
vals, turns, holdings = S.run_track(rule, rets, t0, t1)
bvals = S.bench(rets, t0, t1)

strat_stats, bench_stats = S.stats(vals, turns), S.stats(bvals)
rows = ["CAGR", "Annualized volatility", "Sharpe (rf = 0)", "Maximum drawdown", "Annual one-way turnover", "Cost drag per year", "Growth of $10,000"]
table = pd.DataFrame({"Intermediate momentum (net)": [strat_stats.get(r) for r in rows],
                      "EW benchmark (no costs)": [bench_stats.get(r) for r in rows]}, index=rows)
def fmt(v, r):
    if v is None or (isinstance(v, float) and np.isnan(v)): return "—"
    if r == "Growth of $10,000": return f"${v:,.0f}"
    if r == "Sharpe (rf = 0)": return f"{v:.2f}"
    return f"{v*100:.1f}%"
shown = pd.DataFrame({c: [fmt(table.loc[r, c], r) for r in rows] for c in table.columns}, index=rows)
print(f"In-sample window: {dates[t0].date()} to {dates[t1].date()} ({t1 - t0 + 1} months)")
print(shown.to_string())
table.to_csv(OUT + "/insample_stats_2026-08-31.csv")

In-sample window: 2020-01-31 to 2026-08-31 (80 months)
                        Intermediate momentum (net) EW benchmark (no costs)
CAGR                                          20.3%                   15.4%
Annualized volatility                         23.8%                   21.4%
Sharpe (rf = 0)                                0.85                    0.72
Maximum drawdown                             -21.5%                  -31.7%
Annual one-way turnover                      441.7%                       —
Cost drag per year                             0.9%                       —
Growth of $10,000                           $34,286                 $25,947


In [9]:
# Cell 9 — growth-of-$10,000 exhibit (saved to Drive, Rule 7)
import matplotlib.pyplot as plt
x = dates[t0 - 1 : t1 + 1]                                 # month-ends; the path starts one month before the first holding month
fig, ax = plt.subplots(figsize=(9, 4.8))
ax.plot(x, 10000 * vals, label="Track 1: intermediate momentum (t−12…t−7), top 20 EW, net of 10 bp/side", lw=1.8)
ax.plot(x, 10000 * bvals, label="Equal-weighted panel (200 stocks), no costs", lw=1.8)
ax.axhline(10000, ls="--", lw=0.8, color="gray")
ax.set_title("Growth of $10,000 — in-sample, January 2020 to August 2026 (80 months)")
ax.set_ylabel("Portfolio value ($)")
ax.legend(loc="upper left", fontsize=9)
ax.text(0.01, -0.16, "Source: panel_returns_2026-08-31.csv (Yahoo Finance, dividends reinvested). Panel has survivorship and selection bias; rf = 0 for Sharpe.",
        transform=ax.transAxes, fontsize=7.5, color="gray")
plt.tight_layout()
fig.savefig(OUT + "/insample_growth_2026-08-31.png", dpi=150, bbox_inches="tight")
plt.show()

[Figure displayed here in Colab — the inline PNG preview is omitted from this committed copy to keep the file small; the chart is written to outputs/insample_growth_2026-08-31.png by this cell and regenerates on Runtime → Restart session and run all.]


## Step 7 — hand verification of one month (August 2026)
Pick three of the twenty holdings, look their August 2026 returns up in the CSV, and confirm the arithmetic below: gross = the simple average of the twenty returns; cost = 2 × 0.0010 × turnover; net = gross − cost.

In [10]:
# Cell 10 — one month by hand: the last in-sample month, August 2026 (row 91)
t = t1
hold_now, hold_prev = holdings[t - t0], holdings[t - t0 - 1]
names = sorted(tickers[i] for i in hold_now)
r_month = pd.Series({tickers[i]: rets[t, i] for i in hold_now}).sort_index()
gross = r_month.mean()
turn = len(hold_now - hold_prev) / S.N_HOLD
cost = 2 * S.PER_SIDE * turn
print(f"Month scored: {dates[t].date()} | signal window: {dates[t-12].date()} .. {dates[t-7].date()}")
print("Holdings (20):", ", ".join(names))
print((100 * r_month).round(2).to_string())
print(f"\ngross = mean of the 20 returns = {gross:.6f} ({gross*100:.3f}%)")
print(f"turnover = names replaced / 20 = {len(hold_now - hold_prev)} / 20 = {turn:.2f}  ->  cost = 2 x 0.0010 x {turn:.2f} = {cost:.6f}")
print(f"net = {gross - cost:.6f} ({(gross - cost)*100:.3f}%)  ->  on $10,000: ${10000 * (gross - cost):,.2f}")
print("Three holdings to check against the CSV by hand:", ", ".join(f"{n} {rets[t, tickers.index(n)]*100:+.3f}%" for n in names[:3]))
print("Engine's own value for this month:", f"{vals[t - t0 + 1] / vals[t - t0] - 1:.6f}", "(must equal net above)")

Month scored: 2026-08-31 | signal window: 2025-08-29 .. 2026-01-30
Holdings (20): AMG, AMRX, BTU, FLS, FORM, GAP, GMED, HCC, HP, IONS, LUMN, MSGS, NOV, NYT, OII, ORA, PBF, PLXS, RMBS, VIAV
AMG      0.40
AMRX    -4.64
BTU     39.69
FLS      4.08
FORM    -5.26
GAP     11.05
GMED     3.27
HCC     36.42
HP      26.77
IONS    16.59
LUMN    -2.04
MSGS    -2.27
NOV      9.57
NYT    -10.17
OII      6.11
ORA      4.64
PBF      1.36
PLXS    -4.72
RMBS    -4.77
VIAV    -1.79

gross = mean of the 20 returns = 0.062149 (6.215%)
turnover = names replaced / 20 = 7 / 20 = 0.35  ->  cost = 2 x 0.0010 x 0.35 = 0.000700
net = 0.061449 (6.145%)  ->  on $10,000: $614.49
Three holdings to check against the CSV by hand: AMG +0.401%, AMRX -4.640%, BTU +39.688%
Engine's own value for this month: 0.061449 (must equal net above)


## Step 8 — the briefing-table generator (the only thing Track 2 ever sees)
Format per the Briefing-Table Format Specification: 200 rows sorted by ticker; `Ticker | Company | Sector | 1M | 3M | 12M`, trailing total returns in percent (one decimal) as of the stated month-end. At checkpoint 2 the table is as of **10/30/2026**, the panel's October row (10/31 was a Saturday); it is saved to Drive and pasted, unedited, under the frozen prompt for the November run. (Re-running the generator with `as_of="2026-09-30"` reproduces the checkpoint-1 table byte for byte — the extension changes nothing before October.) QRVO stays in the table, which keeps all 200 rows as the specification requires; if the analyst were to pick it, that pick would be dropped and its weight spread over the other 19, the rules' treatment of a name that has left the universe.


In [11]:
# Cell 11 — briefing table as of a month-end (default: the last row of the panel)
def briefing_table(panel, info, as_of=None):
    r = panel if as_of is None else panel.loc[:as_of]
    as_of = r.index[-1].date()
    r1 = r.iloc[-1]
    r3 = (1 + r.iloc[-3:]).prod() - 1
    r12 = (1 + r.iloc[-12:]).prod() - 1
    tbl = pd.DataFrame({"Ticker": r.columns,
                        "Company": info.loc[r.columns, "Company"].values,
                        "Sector": info.loc[r.columns, "Sector"].values,
                        "1M": (100 * r1.values).round(1) + 0.0, "3M": (100 * r3.values).round(1) + 0.0, "12M": (100 * r12.values).round(1) + 0.0})   # + 0.0 turns -0.0 into 0.0
    return tbl.sort_values("Ticker").reset_index(drop=True), as_of

def briefing_text(tbl, as_of):
    lines = [f"BRIEFING TABLE as of {as_of} (month-end). Trailing total returns in percent, dividends reinvested. {len(tbl)} stocks.",
             "Ticker | Company | Sector | 1M | 3M | 12M"]
    for row in tbl.itertuples(index=False):
        lines.append(f"{row.Ticker} | {row.Company} | {row.Sector} | {row[3]:+.1f} | {row[4]:+.1f} | {row[5]:+.1f}")
    return "\n".join(lines)

tbl, as_of = briefing_table(panel, info)                 # last row of the extended panel = October 2026, dated 10/30 (the checkpoint-2 table)
assert str(as_of) == '2026-10-30', as_of
text = briefing_text(tbl, as_of)
open(OUT + f"/briefing_table_{as_of}.txt", "w").write(text)
tbl.to_csv(OUT + f"/briefing_table_{as_of}.csv", index=False)
print(text)

BRIEFING TABLE as of 2026-10-30 (month-end). Trailing total returns in percent, dividends reinvested. 200 stocks.
Ticker | Company | Sector | 1M | 3M | 12M
ACAD | ACADIA Pharmaceuticals Inc. | Health Care | +2.2 | -20.4 | -9.3
ACM | AECOM | Industrials | +8.0 | -12.0 | -52.1
AD | Array Digital Infrastructure, Inc. | Communication Services | -2.6 | -5.7 | +2.0
ADC | Agree Realty Corporation | Real Estate | +0.8 | -12.8 | -4.1
AGCO | AGCO Corporation | Industrials | -2.9 | +8.9 | +8.6
ALK | Alaska Air Group, Inc. | Industrials | -7.7 | -24.7 | -14.4
AMG | Affiliated Managers Group, Inc. | Financials | +4.4 | +4.0 | +60.4
AMRX | Amneal Pharmaceuticals, Inc. | Health Care | +6.8 | +17.4 | +98.8
AOS | A. O. Smith Corporation | Industrials | +12.6 | +5.5 | -2.2
APPF | AppFolio, Inc. | Information Technology | +6.3 | +18.3 | -16.2
AROC | Archrock, Inc. | Energy | +4.7 | -10.8 | +29.0
ASH | Ashland Inc. | Materials | +10.0 | +3.4 | +57.0
AVA | Avista Corporation | Utilities | +7.1 | -5.9 | +3.

## Checkpoint 2 (Sun 11/8/2026) — Step 10, part 1: rerun the locked code and confirm it reproduces checkpoint 1
October 2026 is row 93. With the extended panel the locked rule's October portfolio is recomputed from the **same** window (October 2025 – March 2026, rows 81–86) and must equal checkpoint 1's Track 1 list to the name. A mismatch means the code or the data vintage changed — the Instructor Guide's cheapest code-integrity check.


In [12]:
# Cell 12 — rerun check: the locked rule's October 2026 holdings from the extended panel vs. checkpoint 1
ENTRY0_T1 = "AMRX, AROC, BTU, CNX, DY, FLS, FORM, FRPT, GMED, HP, IONS, LUMN, MSGS, MTRN, NOV, OII, PSMT, PTEN, SLAB, VIAV".split(", ")
ENTRY0_T2 = "PBF, DK, OII, PTEN, AMRX, HAE, LNTH, CHEF, PSMT, MSGS, STGW, CNK, ZD, MTRN, ASH, EAT, ETSY, AMG, AVT, SLAB".split(", ")   # verbatim (9/15 run)
CP1_T1 = "CWEN, FORM, GMED, HCC, HP, IRDM, KEX, LNTH, MSGS, MTDR, MUR, NOV, NYT, OII, PBF, PTEN, SLAB, TDW, VIAV, WLK".split(", ")
CP1_T2 = "PBF, DK, PTEN, WHD, TDW, SM, HAE, AMRX, RGEN, BIO, BRKR, ICUI, CHEF, EAT, QLYS, AVT, HCC, STGW, ZD, AMG".split(", ")   # verbatim (10/5 run)

months = list(dates.strftime("%Y-%m"))
t_sep, t_oct = months.index("2026-09"), months.index("2026-10")       # rows 92 and 93
oct_rerun = sorted(tickers[i] for i in rule(t_oct, rets))
print(f"October 2026 = row {t_oct}; signal window: {dates[t_oct-12].date()} .. {dates[t_oct-7].date()}")
print("Rerun holdings:", ", ".join(oct_rerun))
match = len(set(oct_rerun) & set(CP1_T1))
print(f"Match with checkpoint 1 Track 1: {match}/20 ->", "OK, code and data vintage unchanged" if oct_rerun == CP1_T1 else "MISMATCH - stop and find out why")
assert oct_rerun == CP1_T1

October 2026 = row 93; signal window: 2025-10-31 .. 2026-03-31
Rerun holdings: CWEN, FORM, GMED, HCC, HP, IRDM, KEX, LNTH, MSGS, MTDR, MUR, NOV, NYT, OII, PBF, PTEN, SLAB, TDW, VIAV, WLK
Match with checkpoint 1 Track 1: 20/20 -> OK, code and data vintage unchanged


## Step 10, part 2 — score October 2026 on the checkpoint-1 holdings
Both tracks held their checkpoint-1 portfolios through October. Each is scored as the equal-weight mean of its 20 holdings' October returns, less 10 bp per side on the names replaced since September (rules file, COSTS): Track 1 replaced 11 names (turnover 0.55, cost 0.11%), Track 2 replaced 9 (turnover 0.45, cost 0.09%). The benchmark is the equal-weighted panel, no costs; QRVO is still active in October (its final partial-month return counts) and leaves the universe from November. The live record compounds September and October.


In [13]:
# Cell 13 — October 2026 scored: Track 1 (locked rule), Track 2 (frozen prompt), EW benchmark; then the live record so far
r_sep, r_oct = pd.Series(rets[t_sep], index=tickers), pd.Series(rets[t_oct], index=tickers)
turn1 = len(set(CP1_T1) - set(ENTRY0_T1)) / S.N_HOLD             # names replaced / 20
turn2 = len(set(CP1_T2) - set(ENTRY0_T2)) / S.N_HOLD
bench_oct = r_oct.mean()                                         # all 200 names are active in October (QRVO through 10/2)
score = pd.DataFrame({
    "gross": [r_oct[CP1_T1].mean(), r_oct[CP1_T2].mean(), bench_oct],
    "turnover": [turn1, turn2, 0.0],
}, index=["Track 1 (locked rule)", "Track 2 (frozen prompt)", "EW benchmark (200, no costs)"])
score["cost"] = 2 * S.PER_SIDE * score["turnover"]
score["net"] = score["gross"] - score["cost"]
score["gap vs benchmark (net)"] = score["net"] - bench_oct
shown = score.copy()
shown[["gross", "cost", "net", "gap vs benchmark (net)"]] *= 100
print(f"Month scored: {dates[t_oct].date()} (row {t_oct}) | percent, except turnover")
print(shown.round(2).to_string())

cost_first = 2 * S.PER_SIDE * 1.0                                # September = first live month, full purchase
live = pd.DataFrame({
    "September net": [r_sep[ENTRY0_T1].mean() - cost_first, r_sep[ENTRY0_T2].mean() - cost_first, r_sep.mean()],
    "October net": score["net"].values,
}, index=score.index)
live["Sep-Oct compounded"] = (1 + live["September net"]) * (1 + live["October net"]) - 1
live["gap vs benchmark"] = live["Sep-Oct compounded"] - live.loc["EW benchmark (200, no costs)", "Sep-Oct compounded"]
print("\nLive record so far, September-October 2026 (percent):")
print((100 * live).round(2).to_string())
print("\nTrack 1 holdings, October returns (%):"); print((100 * r_oct[CP1_T1]).round(1).sort_values().to_string())
print("\nTrack 2 picks, October returns (%):"); print((100 * r_oct[CP1_T2]).round(1).sort_values().to_string())
score.to_csv(OUT + "/live_scores_2026-10.csv")

Month scored: 2026-10-30 (row 93) | percent, except turnover
                              gross  turnover  cost   net  gap vs benchmark (net)
Track 1 (locked rule)          2.44      0.55  0.11  2.33                   -3.31
Track 2 (frozen prompt)        4.91      0.45  0.09  4.82                   -0.82
EW benchmark (200, no costs)   5.64      0.00  0.00  5.64                    0.00

Live record so far, September-October 2026 (percent):
                              September net  October net  Sep-Oct compounded  gap vs benchmark
Track 1 (locked rule)                 -3.63         2.33               -1.39             -0.91
Track 2 (frozen prompt)               -0.84         4.82                3.94              4.41
EW benchmark (200, no costs)          -5.78         5.64               -0.47              0.00

Track 1 holdings, October returns (%):
FORM   -41.4
VIAV   -28.6
OII     -6.3
IRDM    -6.2
HCC     -2.4
SLAB     0.4
LNTH     0.9
PBF      1.4
MSGS     2.5
NOV      3.0
GMED  

## Step 10, part 3 — November 2026 holdings for Track 1 (data through 10/30)
November is row 94, one past the extended panel, so the window is November 2025 – April 2026 (rows 82–87). QRVO has left the universe (acquired 10/5/2026). Its window is complete, so the locked code still ranks it, but the rules file governs: an exited name may not be held. QRVO ranks 135th, so the code's list and the rule's list coincide, and the assert below documents it.


In [14]:
# Cell 14 — the locked rule's November 2026 portfolio, from data through October 2026
EXITED = ["QRVO"]                                                # acquired by Skyworks 10/5/2026 (last trade 10/2): out of the universe from November
t_next = len(dates)                                              # row 94 = November 2026
hold_nov = rule(t_next, rets)
nov_list = sorted(tickers[i] for i in hold_nov)
assert not set(nov_list) & set(EXITED), "an exited name was selected: drop it and take the next eligible name (rules: UNIVERSE)"
sig_next = pd.Series(S.signal(t_next, rets), index=tickers)
print(f"Signal window: {dates[t_next-12].date()} .. {dates[t_next-7].date()} (six months)")
print("TRACK 1 November 2026 holdings (alphabetical):")
print(", ".join(nov_list))
replaced = sorted(set(nov_list) - set(CP1_T1)); dropped = sorted(set(CP1_T1) - set(nov_list))
turn_nov = len(replaced) / S.N_HOLD
print(f"\nNames replaced vs October: {len(replaced)} of 20 -> November one-way turnover {turn_nov:.2f}, cost 2 x 0.0010 x {turn_nov:.2f} = {2*S.PER_SIDE*turn_nov*100:.2f}% (charged when November is scored)")
print("Added:  ", ", ".join(replaced)); print("Dropped:", ", ".join(dropped))
ranked = sig_next.sort_values(ascending=False)
print("\nRanks 18-23 around the boundary (signal = compounded 6-month return, %):")
print((100 * ranked.iloc[17:23]).round(2).to_string())
print(f"\nQRVO on the locked signal: rank {list(ranked.index).index('QRVO') + 1} of 200 (exited, so ineligible in any case)")
with open(OUT + "/track1_holdings_2026-11.txt", "w") as f:
    f.write(", ".join(nov_list) + "\n")

Signal window: 2025-11-28 .. 2026-04-30 (six months)
TRACK 1 November 2026 holdings (alphabetical):
AROC, AVT, ESI, FORM, HP, IRDM, MSGS, MTDR, MTRN, MUSA, OII, PLXS, PTEN, R, SAIA, SLAB, SM, TDW, VIAV, WLK

Names replaced vs October: 9 of 20 -> November one-way turnover 0.45, cost 2 x 0.0010 x 0.45 = 0.09% (charged when November is scored)
Added:   AROC, AVT, ESI, MTRN, MUSA, PLXS, R, SAIA, SM
Dropped: CWEN, GMED, HCC, KEX, LNTH, MUR, NOV, NYT, PBF

Ranks 18-23 around the boundary (signal = compounded 6-month return, %):
SAIA    53.44
SM      51.41
R       51.40
MUR     50.72
GMED    49.33
LNTH    46.68

QRVO on the locked signal: rank 135 of 200 (exited, so ineligible in any case)


## Step 10, part 4 — the pre-registered placebo, two months in (a preview, not the test)
The same 1,000 draws as at checkpoint 1 (numpy `default_rng(4075)`, 20 names drawn without replacement from the 200 active at the start of the live window), bought and held: September counts as a full purchase (cost 0.20%) and October costs nothing (no names replaced). QRVO was still active in October, so draws that hold it keep its partial-month return; from November it is dropped and its weight redistributed (rules file, PLACEBO PLAN). The gap is the compounded September–October return minus the benchmark's, the same shape as the December test.


In [15]:
# Cell 15 — placebo distribution of two-month gaps (same draws the final test will use)
rng = np.random.default_rng(4075)
draws = [rng.choice(len(tickers), size=S.N_HOLD, replace=False) for _ in range(1000)]
bench_2m = live.loc["EW benchmark (200, no costs)", "Sep-Oct compounded"]
placebo_2m = np.array([(1 + rets[t_sep, d].mean() - cost_first) * (1 + rets[t_oct, d].mean()) - 1 for d in draws])   # buy and hold: no turnover in October
gaps = placebo_2m - bench_2m
t1_gap, t2_gap = live.loc["Track 1 (locked rule)", "gap vs benchmark"], live.loc["Track 2 (frozen prompt)", "gap vs benchmark"]
pct = lambda g: 100 * (gaps < g).mean()
print(f"Placebo gaps vs benchmark, September-October compounded (1,000 portfolios): 5th {100*np.percentile(gaps, 5):+.2f}pp | median {100*np.percentile(gaps, 50):+.2f}pp | 95th {100*np.percentile(gaps, 95):+.2f}pp | range {100*gaps.min():+.2f} .. {100*gaps.max():+.2f}pp")
print(f"Track 1 gap {100*t1_gap:+.2f}pp -> {pct(t1_gap):.1f}th percentile | Track 2 gap {100*t2_gap:+.2f}pp -> {pct(t2_gap):.1f}th percentile")
print(f"Draws holding QRVO: {sum(tickers.index('QRVO') in d for d in draws)} of 1,000 (its partial October return counts; dropped from November)")
print("Two months of 20-stock portfolios span roughly -4 to +3.5pp around the benchmark at the 5th-95th percentiles; nothing is concluded until the three-month test in December.")

Placebo gaps vs benchmark, September-October compounded (1,000 portfolios): 5th -3.90pp | median -0.13pp | 95th +3.45pp | range -6.66 .. +6.74pp
Track 1 gap -0.91pp -> 37.0th percentile | Track 2 gap +4.41pp -> 98.3th percentile
Draws holding QRVO: 115 of 1,000 (its partial October return counts; dropped from November)
Two months of 20-stock portfolios span roughly -4 to +3.5pp around the benchmark at the 5th-95th percentiles; nothing is concluded until the three-month test in December.


## Step 10, part 5 — Track 2: the frozen prompt's November picks, pasted verbatim
The run happens outside the notebook, in a new empty claude.ai chat on the frozen model and settings (`llm_analyst_prompt.md`): the prompt, one blank line, then the 10/30 briefing table from Cell 11, unedited. The reply is pasted below exactly as returned (the log's chain of custody). The cell checks the format rule (exactly 20 distinct tickers from the table), flags any pick that has left the universe, and measures the overlap with Track 1's November list.


In [16]:
# Cell 16 — Track 2 November 2026 picks (verbatim paste) and the format check
T2_RUN = "11/2/2026, 5:31 PM Central (Mon)"      # date/time of the single sanctioned run (Central time)
T2_RAW = """HAE, AMRX, ITGR, LNTH, PBF, DK, PTEN, SM, CVI, QLYS, SAIC, SLAB, CVLT, STGW, IRDM, MSGS, EAT, CROX, TTC, AVNT"""   # the model's reply, pasted verbatim - do not retype, reorder or clean

import re
toks = [t for t in re.split(r"[,\s]+", T2_RAW.strip()) if t]
valid = [t for t in toks if t in tickers]
print(f"Run: {T2_RUN} | tokens returned: {len(toks)} | valid panel tickers: {len(valid)} | distinct: {len(set(valid))}")
malformed = not (len(toks) == 20 and len(set(valid)) == 20 and T2_RAW.strip().replace(" ", "").count(",") == 19)
print("Format rule:", "MALFORMED - one logged retry allowed (new chat, identical message)" if malformed else "OK - exactly 20 distinct tickers from the table, no other text; parsed on the first try")
nov_t2 = toks
exited = sorted(set(nov_t2) & set(EXITED))
print("Picks that have left the universe:", ", ".join(exited) if exited else "none")
overlap = sorted(set(nov_t2) & set(nov_list))
print(f"Overlap with Track 1's November list: {len(overlap)}/20 ({', '.join(overlap) if overlap else 'none'})")
t2_turn_nov = len(set(nov_t2) - set(CP1_T2)) / S.N_HOLD
print(f"Names replaced vs Track 2's October picks: {len(set(nov_t2) - set(CP1_T2))} of 20 (turnover {t2_turn_nov:.2f}, charged when November is scored)")
print("Sectors of the picks:"); print(info.loc[nov_t2, "Sector"].value_counts().to_string())
with open(OUT + "/track2_picks_2026-11.txt", "w") as f:
    f.write(T2_RAW.strip() + "\n")

Run: 11/2/2026, 5:31 PM Central (Mon) | tokens returned: 20 | valid panel tickers: 20 | distinct: 20
Format rule: OK - exactly 20 distinct tickers from the table, no other text; parsed on the first try
Picks that have left the universe: none
Overlap with Track 1's November list: 5/20 (IRDM, MSGS, PTEN, SLAB, SM)
Names replaced vs Track 2's October picks: 11 of 20 (turnover 0.55, charged when November is scored)
Sectors of the picks:
Sector
Energy                    5
Health Care               4
Information Technology    4
Communication Services    3
Consumer Discretionary    2
Industrials               1
Materials                 1


## Before the checkpoint-2 commit (Sun 11/8)
1. Runtime → **Restart session and run all** — no errors (Cell 12's assert is the rerun check), then save.
2. Append the CHECKPOINT 2 entry to `monitoring_log.md`: the rerun match (20/20), October scored for both tracks and the benchmark with the live record so far (Cell 13), November holdings for both tracks (Cells 14 and 16; the LLM's list verbatim), the QRVO exit, 3–5 sentences of attribution (Cell 15 gives the placebo span), the pre-registration hash quoted.
3. Commit to FINTECH-P2-Example: `panel_returns_2026-10-31.csv` (hash in Cell 2), this notebook under its dated name (File → **Save a copy in GitHub**; keep the 9/20 original and the checkpoint-1 copy), `monitoring_log.md`. Nothing else changes — not the rules, not the code, not the prompt.
